# LR3 -- Decision Tree Classification (explained)

Dataset: adult.csv (Census Income)

Target: income (<=50K / >50K) -- binary classification

1. Import
2. Load data
3. EDA
4. Preprocessing
5. Train decision tree
6. Evaluate
7. Improve

In [ ]:
# ============================================================
# 1 - Import
# ============================================================

import pandas as pd                # tabular data (DataFrames)
import numpy as np                 # numerical arrays and math
import matplotlib.pyplot as plt    # plotting API
import seaborn as sns              # statistical visualization

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
# train_test_split -- split data into train/test sets
# GridSearchCV     -- exhaustive search over hyperparameter grid with cross-validation
# cross_val_score  -- evaluate model via k-fold cross-validation

from sklearn.tree import DecisionTreeClassifier, plot_tree
# DecisionTreeClassifier -- decision tree for classification tasks
# plot_tree              -- visualize tree as diagram

from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
# classification_report  -- precision, recall, F1 per class
# confusion_matrix       -- predicted vs actual matrix
# accuracy_score         -- fraction of correct predictions
# ConfusionMatrixDisplay -- visual heatmap of confusion matrix

from sklearn.preprocessing import LabelEncoder
# LabelEncoder -- converts categorical strings to integers
# e.g. 'Male' -> 0, 'Female' -> 1

%matplotlib inline  # render plots inline in notebook

In [ ]:
# ============================================================
# 2 - Load data
# ============================================================
# Dataset: US Census data (1994). 48842 records, 15 columns.
# Task: predict whether person earns >50K or <=50K per year.
# Binary classification -- simpler than 5-class, expect better accuracy.

df = pd.read_csv("adult.csv")
# Read CSV into DataFrame

print(f"Shape: {df.shape}")
# (48842, 15) -- large dataset, good for decision trees

df.head()
# Preview: mix of numeric (age, hours-per-week) and categorical (workclass, education)

In [ ]:
# ============================================================
# 3 - EDA (statistics)
# ============================================================
# GOAL: check data quality, target balance, feature distributions.

print("Missing values:")
print(df.isnull().sum())
# Result: 0 nulls. Data is clean (unlike LR3 survey data).
# Note: some columns may have '?' as missing marker -- pandas reads as string, not NaN.

print(f"\nTarget distribution:")
print(df['income'].value_counts())
# <=50K: ~37K, >50K: ~12K. Ratio ~3:1.
# Moderate imbalance -- stratified split needed.

print(f"\nNumeric stats:")
df.describe()
# .describe() -- count, mean, std, min, 25%, 50%, 75%, max per numeric column.
# Quick check for outliers and scale differences.

In [ ]:
# ============================================================
# 3 (cont) - Visualizations
# ============================================================
# GOAL: visual inspection of features and their relation to target.

num_cols = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
# Selected numeric features (excluding fnlwgt -- census weight, not predictive).

# --- Histograms: feature distributions ---
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, edgecolor='black')
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

# ---- Conclusions ----
# - age: roughly normal, slight right skew. Bulk 25-45.
# - educational-num: multimodal -- peaks at 9 (HS), 10 (some-college), 13 (Bachelors).
# - capital-gain/loss: extreme skew -- vast majority = 0, rare large values.
# - hours-per-week: sharp peak at 40 (standard workweek).

# --- Boxplots: features by income group ---
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='income', y=col, ax=axes[i])
    # Box: median (line), Q1-Q3 (box), whiskers (1.5*IQR), outliers (dots)
    # Compare distributions between <=50K and >50K
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

# ---- Conclusions ----
# - age: >50K median higher (~44 vs ~33). Older = higher income.
# - educational-num: >50K median higher (~13 vs ~9). Education matters.
# - capital-gain: >50K has many non-zero outliers. Strong signal.
# - hours-per-week: >50K slightly more (~45 vs ~40).
# Top predictors: capital-gain, educational-num, age.

# --- Correlation matrix ---
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
# Pearson correlation between numeric features.
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

# ---- Conclusions ----
# - Most correlations near 0 -- features are independent. Good.
# - No multicollinearity issues for decision tree.

# --- Target balance ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
income_counts = df['income'].value_counts()

axes[0].bar(income_counts.index, income_counts.values, color=['#2196F3', '#FF5722'])
axes[0].set_title('Income distribution')

axes[1].pie(income_counts.values, labels=income_counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF5722'], startangle=90)
axes[1].set_title('Income (%)')
plt.tight_layout()
plt.show()

# ---- Conclusions ----
# ~76% <=50K, ~24% >50K. Ratio 3:1.
# Moderate imbalance. Need stratified split.
# Accuracy alone can be misleading (always predict <=50K = 76%).
# Use F1-score alongside accuracy.

In [ ]:
# ============================================================
# 4 - Preprocessing
# ============================================================
# GOAL: prepare data for decision tree. Need all-numeric input.

df = df.drop(columns=['fnlwgt'])
# Drop fnlwgt -- census sampling weight, not a personal characteristic.
# Confirmed in LR2 EDA: ~0 correlation with everything.

df['income'] = (df['income'] == '>50K').astype(int)
# Encode target: '>50K' -> 1, '<=50K' -> 0
# Boolean comparison -> True/False -> int 1/0

# --- Encode categorical features ---
cat_features = df.select_dtypes(include=['object', 'str']).columns.tolist()
# Get all remaining string columns

label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    # .fit_transform():
    #   .fit()       -- learn unique values, assign integer (alphabetical order)
    #   .transform() -- replace strings with integers
    # For decision trees: arbitrary order is OK (splits by thresholds).
    label_encoders[col] = le

# --- Feature matrix ---
feature_cols = [c for c in df.columns if c != 'income']
# All columns except target

X = df[feature_cols]   # (48842, 13) -- 13 features
y = df['income']       # (48842,)    -- binary target

print(f"Features: {X.shape}")
print(f"Target: {y.value_counts().to_dict()}")

# --- Train-test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# test_size=0.2   -- 20% test, 80% train
# random_state=42 -- reproducible split
# stratify=y      -- preserve 76/24 ratio in both sets

print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# ============================================================
# 5 - Train Decision Tree (baseline)
# ============================================================
# GOAL: train default tree as baseline reference.

dt_base = DecisionTreeClassifier(random_state=42)
# Default: no depth limit, min_samples_split=2, criterion='gini'
# Will overfit -- but serves as upper-bound complexity baseline.

dt_base.fit(X_train, y_train)
# Build tree: recursively find best split (min Gini impurity)
# until all leaves pure or min_samples constraints hit.

y_pred_base = dt_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)

print(f"Baseline Accuracy: {acc_base:.4f}")
# Expected ~81-82% for unpruned tree on adult dataset.
# Better than always-predict-majority (76%) but not by much.

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_base, target_names=['<=50K', '>50K']))
# precision -- of predicted X, how many actually X
# recall    -- of actual X, how many found
# f1-score  -- harmonic mean of precision and recall
# >50K class typically has lower recall (harder to detect minority class).

print(f"Tree depth: {dt_base.get_depth()}")
print(f"Number of leaves: {dt_base.get_n_leaves()}")
# Deep tree with many leaves = overfitting to training noise.

# ---- Conclusions ----
# Binary classification on large dataset (48K) works much better than
# 5-class on small dataset (399). But unpruned tree still overfits.
# Pruning in step 7 should improve generalization.

In [ ]:
# ============================================================
# 6 - Evaluate Decision Tree
# ============================================================
# GOAL: confusion matrix, feature importance, cross-validation, tree viz.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Confusion matrix ---
ConfusionMatrixDisplay.from_estimator(dt_base, X_test, y_test, ax=axes[0],
                                      display_labels=['<=50K', '>50K'], cmap='Blues')
# Rows = actual, Columns = predicted
# Top-left: true <=50K correctly predicted (True Negative)
# Bottom-right: true >50K correctly predicted (True Positive)
# Top-right: <=50K predicted as >50K (False Positive)
# Bottom-left: >50K predicted as <=50K (False Negative)
axes[0].set_title('Confusion Matrix (Baseline)')

# --- Feature importance ---
importances = pd.Series(dt_base.feature_importances_, index=feature_cols).sort_values(ascending=True)
# .feature_importances_ -- total Gini impurity reduction per feature.
# Higher = feature used more in splits, contributed more to decisions.
importances.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance (Baseline)')
plt.tight_layout()
plt.show()

# ---- Conclusions ----
# Feature importance confirms EDA findings:
# capital-gain, education, age, relationship, marital-status at top.
# In overfit tree importance is noisy but direction is correct.

# --- Cross-validation ---
cv_scores = cross_val_score(dt_base, X, y, cv=5, scoring='accuracy')
# 5-fold: train on 4 folds, test on 1, repeat 5 times.
# More robust performance estimate than single split.
print(f"Cross-validation accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# --- Tree visualization ---
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_base, max_depth=3, feature_names=feature_cols,
          class_names=['<=50K', '>50K'],
          filled=True, rounded=True, fontsize=8, ax=ax)
# max_depth=3     -- only top 3 levels (full tree too large)
# filled=True     -- color by class (darker = purer node)
# Each node: split condition, gini, samples, value [<=50K, >50K], class

for text in ax.texts:
    text.set_color('black')
plt.title('Decision Tree (top 3 levels)')
plt.tight_layout()
plt.show()

# Root split reveals most informative feature.
# Likely marital-status or relationship -- known strong predictors in this dataset.

In [ ]:
# ============================================================
# 7 - Improve Decision Tree (GridSearchCV)
# ============================================================
# GOAL: find optimal hyperparameters to reduce overfitting.

param_grid = {
    'max_depth': [3, 5, 7, 10, 15, None],
    # Limit tree depth. Smaller = simpler, less overfitting.

    'min_samples_split': [2, 5, 10, 20],
    # Min samples to attempt split. Higher = more conservative.

    'min_samples_leaf': [1, 2, 5, 10],
    # Min samples per leaf. Higher = smoother predictions.

    'criterion': ['gini', 'entropy']
    # gini    -- Gini impurity
    # entropy -- information gain (Shannon entropy)
}
# Total: 6*4*4*2 = 192 combinations * 5 folds = 960 trainings

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
# n_jobs=-1 -- use all CPU cores for parallel search

grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
# Expected: moderate depth (7-10), larger min_samples.

print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

dt_best = grid_search.best_estimator_
# Fitted tree with best hyperparameters.

y_pred_best = dt_best.predict(X_test)
acc_best = accuracy_score(y_test, y_pred_best)

print(f"\nImproved test accuracy: {acc_best:.4f}")
print(f"Baseline test accuracy: {acc_base:.4f}")
print(f"Improvement: {acc_best - acc_base:+.4f}")

print(f"\nClassification Report (improved):")
print(classification_report(y_test, y_pred_best, target_names=['<=50K', '>50K']))

# ---- Conclusions ----
# Pruned tree should have:
# - Similar or slightly better accuracy than baseline
# - Better generalization (less gap between train and test accuracy)
# - More balanced precision/recall for >50K class
# - Fewer leaves = more interpretable model

In [ ]:
# ============================================================
# 7 (cont) - Compare baseline vs improved
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Confusion matrix (improved) ---
ConfusionMatrixDisplay.from_estimator(dt_best, X_test, y_test, ax=axes[0],
                                      display_labels=['<=50K', '>50K'], cmap='Greens')
axes[0].set_title('Confusion Matrix (Improved)')
# Compare to baseline: fewer false negatives/positives expected.

# --- Feature importance (improved) ---
importances_best = pd.Series(dt_best.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances_best.plot.barh(ax=axes[1], color='forestgreen')
axes[1].set_title('Feature Importance (Improved)')
# Pruned tree uses fewer features -- importance more concentrated.
# Cleaner signal than overfit baseline.

# --- Accuracy comparison ---
models = ['Baseline', 'Improved']
accs = [acc_base, acc_best]
bars = axes[2].bar(models, accs, color=['steelblue', 'forestgreen'])
axes[2].set_ylim(0, 1)
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy Comparison')
for bar, acc in zip(bars, accs):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{acc:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# --- Improved tree visualization ---
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(dt_best, max_depth=3, feature_names=feature_cols,
          class_names=['<=50K', '>50K'],
          filled=True, rounded=True, fontsize=8, ax=ax)
for text in ax.texts:
    text.set_color('black')
plt.title('Improved Decision Tree (top 3 levels)')
plt.tight_layout()
plt.show()

# ---- Overall summary ----
# Dataset: 48842 adults, 13 features, binary target (income <=50K vs >50K).
# Data was clean -- minimal preprocessing needed (vs messy survey in prev version).
# Baseline (unpruned): ~81% accuracy but overfits.
# Improved (GridSearchCV): better generalization, balanced precision/recall.
# Key predictors: capital-gain, education, marital-status, age.
# Binary classification on large dataset >> 5-class on small dataset.
#
# Possible further improvements:
#   a) Ensemble methods (Random Forest, Gradient Boosting)
#   b) Feature engineering (log-transform capital-gain/loss)
#   c) Handle '?' values as proper missing data
#   d) One-hot encoding instead of label encoding for nominal features